In [ ]:
import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import pickle

from multiprocessing import Pool
from tqdm import tqdm

import os
import sys

sys.path.append('/exp/sbnd/app/users/munjung/calibration/ana/pyana')
from pandas_helpers import *
from branches import *

sys.path.append('/exp/sbnd/app/users/munjung/pyana_utils')
from particle_energies import *
from constants import *
from sbnd_configs import *
from flatcaf_utils import *

# plt.style.use('./presentation.mplstyle')

# ignore FutureWarning from uproot
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning) # df2 = df.sort_index()

In [ ]:
# save fig?
save = False

In [ ]:
MASS_PROTON = pdg["proton"][2]
MASS_MUON = pdg["muon"][2]

In [ ]:
def load_data(data_filename, plane=2):
    events = uproot.open(data_filename+":recTree")

    hdrdf = loadbranches(events, hdrbranches)
    run = hdrdf.rec.hdr.run
    subrun = hdrdf.rec.hdr.subrun
    evt = hdrdf.rec.hdr.evt

    slcdf = loadbranches(events, slcbranches)
    slcdf = slcdf #.rec #.slc

    trkdf = loadbranches(events, trkbranches)
    truetrkdf = loadbranches(events, truetrkbranches)
    trkdf = trkdf.rec.slc.reco.pfp #.trk
    truetrkdf = truetrkdf.rec.slc.reco.pfp.trk
    trkdf = trkdf.join(truetrkdf)

    trkhitbranches = trkhitbranches_perplane(plane)
    trkhitbranches += [
        trkbranch + "calo.%i.points.phi"% plane,
        trkbranch + "calo.%i.points.efield"% plane,
    ]
    hitdf = loadbranches(events, trkhitbranches)
    hitdf = hitdf.rec.slc.reco.pfp.trk.calo

    masterdf = pd.merge(slcdf.reset_index(), 
                        trkdf.reset_index(),
                        left_on=[("entry", "", ""), ("rec.slc..index", "", "")], 
                        right_on=[("entry", "", "" ), ("rec.slc..index", "", "")], 
                        how="right", # Keep every track
                        )
    masterdf = masterdf.set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index"], verify_integrity=True)

    run.name = ("run", "", "", "")
    subrun.name = ("subrun", "", "", "")
    evt.name = ("evt", "", "", "")
    masterdf = masterdf.join(run)
    masterdf = masterdf.join(subrun)
    masterdf = masterdf.join(evt)
    hitdf = hitdf.join(run)
    hitdf = hitdf.join(subrun)
    hitdf = hitdf.join(evt)

    hitdf = pd.merge(trkdf.reset_index(), hitdf.reset_index(),
            left_on=["entry", "rec.slc..index", "rec.slc.reco.pfp..index"],
            right_on=["entry", "rec.slc..index", "rec.slc.reco.pfp..index"],
            how="right")
    hitdf = hitdf.set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index", "rec.slc.reco.pfp.trk.calo.{}.points..index".format(plane)], verify_integrity=True)

    # hitdf = masterdf.join(hitdf)
    # masterdf = masterdf.join(hitdf)
    hitdf = hitdf
    slcdf = masterdf.sort_values(("trk","len"), ascending=False).groupby(level=[0,1]).first()

    events.close()
    return masterdf, slcdf, hitdf

## Proton Selections

In [ ]:
## selection
def select_topology(df, nprong=2, max_len_cut=25, min_len_cut=25, contained=True):
    df = df[(InFV(df.rec.slc.vertex)) & (df.trk.len > 0)]

    npfps = df.groupby(level=[0,1]).count().rec.slc.producer
    twopfps_idx = npfps[npfps == nprong].index 
    twopfps = df.reset_index(level=[2]).loc[twopfps_idx]
    twopfps = twopfps.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index"])

    twopfps["is_track"] = (twopfps.trackScore > 0.5)
    twopfps["is_contained_track"] = (twopfps.trackScore > 0.5) & (InFV(twopfps.trk.start) & InFV(twopfps.trk.end))
    if contained:
        ntrks = twopfps.groupby(level=[0,1]).is_contained_track.sum()
    else:
        ntrks = twopfps.groupby(level=[0,1]).is_track.sum()

    twoprong_slc_idx = ntrks[(ntrks == nprong)].index
    # twoprong_slc_idx = ntrks[(ntrks >= 2)].index
    twoprong = df.reset_index().set_index(["entry", "rec.slc..index"]).loc[twoprong_slc_idx]
    twoprong = twoprong.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index"])

    maxlen = twoprong.groupby(level=[0,1]).max().trk.len
    minlen = twoprong.groupby(level=[0,1]).min().trk.len
    longtrack_slc_idx = maxlen[(maxlen > max_len_cut) & (minlen > min_len_cut)].index

    longer_df = twoprong.sort_values(by=("trk","len","", ""), ascending=False).groupby(level=[0,1]).nth(0)
    shorter_df = twoprong.sort_values(by=("trk", "len", "", ""), ascending=True).groupby(level=[0,1]).nth(0)
    return longer_df, shorter_df

def select_calorimetry(df, muscore_cut_val=20, pscore_cut_val=80, pid=2212):
    if pid == 2212:
        muscore_cut = (df.trk.chi2pid.I2.chi2_muon > muscore_cut_val)
        pscore_cut = (df.trk.chi2pid.I2.chi2_proton < pscore_cut_val)

    elif pid == 13:
        muscore_cut = (df.trk.chi2pid.I2.chi2_muon < muscore_cut_val)
        pscore_cut = (df.trk.chi2pid.I2.chi2_proton > pscore_cut_val)

    score_selected = df[muscore_cut & pscore_cut]

    return score_selected

In [ ]:
def check_flipped(hitdf, plane=2):
    trk_hitdfs = []
    hitdf["flipped"] = False

    trks = hitdf.reset_index(level=[3]).index.unique()
    for pidx in range(len(trks)):
        this_trk_hits = hitdf.reset_index(level=[3]).loc[trks[pidx]]

        if len(this_trk_hits) == 0:
            continue

        first_half = this_trk_hits.iloc[:len(this_trk_hits)//2]
        last_half = this_trk_hits.iloc[len(this_trk_hits)//2:]
        first_avg = np.mean(first_half[("I{}".format(plane), "points", "dqdx", "")])
        last_avg = np.mean(last_half[("I{}".format(plane), "points", "dqdx", "")])
        if first_avg > last_avg:
            max_rr = this_trk_hits[("I{}".format(plane), "points", "rr", "")].max()
            this_trk_hits[("I{}".format(plane), "points", "rr", "")] = max_rr - this_trk_hits[("I{}".format(plane), "points", "rr", "")]
            this_trk_hits["flipped"] = True

        trk_hitdfs.append(this_trk_hits)

    hitdf = pd.concat(trk_hitdfs)
    hitdf = hitdf.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index", "rec.slc.reco.pfp.trk.calo.{}.points..index".format(plane)])
    return hitdf

In [ ]:
def get_muon_hits(infile, plane=2, nprong=1):

    masterdf, slcdf, hitdf = load_data(infile, plane)

    # 2 prong    
    longer_data, shorter_data = select_topology(masterdf, nprong=2, max_len_cut=25, min_len_cut=25, contained=True)
    muon_candidate = select_calorimetry(longer_data, muscore_cut_val=20, pscore_cut_val=80, pid=13)

    muon_hits = hitdf.reset_index(level=[3]).loc[muon_candidate.index]
    muon_hits = muon_hits.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index", "rec.slc.reco.pfp.trk.calo.{}.points..index".format(plane)])
    if len(muon_hits) == 0:
        return None
        
    return muon_hits
    

def get_proton_hits(infile, plane=2, nprong=1):

    masterdf, slcdf, hitdf = load_data(infile, plane)

    # 1 prong    
    longer_data, shorter_data = select_topology(masterdf, nprong=1, max_len_cut=25, min_len_cut=25, contained=True)
    proton_candidate = select_calorimetry(shorter_data, muscore_cut_val=20, pscore_cut_val=80)

    proton_hits = hitdf.reset_index(level=[3]).loc[proton_candidate.index]
    proton_hits = proton_hits.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index", "rec.slc.reco.pfp.trk.calo.{}.points..index".format(plane)])
    if len(proton_hits) == 0:
        return None
        
    proton_hits_1 = check_flipped(proton_hits, plane=plane)
    proton_hits_1["selection"] = 1

    # 2 prong    
    longer_data, shorter_data = select_topology(masterdf, nprong=2, max_len_cut=25, min_len_cut=25, contained=True)
    proton_candidate = select_calorimetry(shorter_data, muscore_cut_val=20, pscore_cut_val=80)

    proton_hits = hitdf.reset_index(level=[3]).loc[proton_candidate.index]
    proton_hits = proton_hits.reset_index().set_index(["entry", "rec.slc..index", "rec.slc.reco.pfp..index", "rec.slc.reco.pfp.trk.calo.{}.points..index".format(plane)])
    if len(proton_hits) == 0:
        return None
        
    proton_hits_2 = check_flipped(proton_hits, plane=plane)
    proton_hits_2["selection"] = 2

    proton_hits = pd.concat([proton_hits_1, proton_hits_2], ignore_index=False) 

    return proton_hits
    

In [ ]:
listname = "filelists/2025B_dev_flatcafs.list"
save_tag = "2025B_dev_data" # name to save the output
infiles = []
with open(listname, "r") as f:
    for fname in f:
        if fname.startswith("/pnfs"):
            fname = fname.replace("/pnfs", "root://fndcadoor.fnal.gov:1094/pnfs/fnal.gov/usr")

        infiles.append(fname[:-1])

print(len(infiles), "file")
print(infiles[:3])

# infiles = infiles[:10]
hit_list = []
with Pool(processes=60) as pool:
    for hits in tqdm(pool.imap_unordered(get_proton_hits, infiles), total=len(infiles)):
        if hits is not None:
            hit_list.append(hits)
        else:
            # print("error on file")
            continue

In [ ]:
# add all 
proton_hits = pd.concat(hit_list)
print(len(proton_hits))
proton_hits.head()

In [ ]:
nbins = 101
rrbins = np.linspace(0,150,nbins)
dqdxbins = np.linspace(0,0.85e4,nbins)

fig, ax = plt.subplots(1,2, figsize=(14,5))
bins = [rrbins, dqdxbins]
ax[0].hist2d(proton_hits.I2.points.rr, proton_hits.I2.points.dqdx, bins=bins, norm=LogNorm())
ax[0].set_xlabel("Residual Range [cm]")
ax[0].set_ylabel("dQ/dx [ADC/cm]")

cut = (proton_hits.flipped == False)
cut = (proton_hits.truth.p.pdg ==2212)
ax[1].hist2d(proton_hits[cut].I2.points.rr, proton_hits[cut].I2.points.dqdx, bins=bins, norm=LogNorm())
ax[1].set_xlabel("Residual Range [cm]")
ax[1].set_ylabel("dQ/dx [ADC/cm]")

if save:
    plt.savefig("figures/proton_selection/dqdx_rr.pdf", bbox_inches="tight")
plt.show();

# Hit Selection

In [ ]:
def drop_endhits(hitdf, plane=2, ndrop=3):
    trk_hitdfs = []

    trks = hitdf.reset_index(level=[3]).index.unique()
    for pidx in tqdm(range(len(trks))):
        this_trk_hits = hitdf.reset_index(level=[3]).loc[trks[pidx]]
        this_trk_hits = this_trk_hits.sort_values(by=("I{}".format(plane), "points", "rr", ""), ascending=False)
        this_trk_hits = this_trk_hits.iloc[ndrop:]

        trk_hitdfs.append(this_trk_hits)

    hitdf = pd.concat(trk_hitdfs)
    return hitdf

In [ ]:
def check_badorder(df, plane=2):
    # df = hitdf.set_index(["run", "subrun", "event"])

    trk_hitdfs = []
    df["badorder"] = False

    trks = df.index.unique()
    for pidx in tqdm(range(len(trks))):
        this_trk_hits = df.loc[trks[pidx]] #.reset_index()

        idx_rr = this_trk_hits.sort_values(by=("I{}".format(plane), "points", "rr", ""), ascending=False).index
        idx_wire = this_trk_hits.sort_values(by=("I{}".format(plane), "points", "wire", ""), ascending=False).index
        if (idx_rr[0] > idx_rr[-1]):
            idx_rr = idx_rr[::-1]
        if (idx_wire[0] > idx_wire[-1]):
            idx_wire = idx_wire[::-1]

        if ((idx_rr != idx_wire).any()):
            this_trk_hits["badorder"] = True

        trk_hitdfs.append(this_trk_hits)

    df = pd.concat(trk_hitdfs)
    return df

In [ ]:
def check_ntpcs(df, plane=2):
    # df = hitdf.set_index(["run", "subrun", "event"])

    trk_hitdfs = []
    df["ntpcs"] = False

    trks = df.index.unique()
    for pidx in tqdm(range(len(trks))):
        this_trk_hits = df.loc[trks[pidx]] #.reset_index()

        if len(this_trk_hits) == 0:
            continue

        tpcs = this_trk_hits.I2.points.tpc.unique()

        if (len(tpcs) == 2):
            this_trk_hits["ntpcs"] = 2
        elif (len(tpcs) == 1):
            this_trk_hits["ntpcs"] = 1
        else:
            raise ValueError(f"Number of TPCs is {len(tpcs)}")

        trk_hitdfs.append(this_trk_hits)

    df = pd.concat(trk_hitdfs)
    return df

In [ ]:
def check_wireskip(df, plane=2):
    # df = hitdf.set_index(["run", "subrun", "event"])

    trk_hitdfs = []
    df["wireskip"] = 0

    trks = df.index.unique()
    for pidx in tqdm(range(len(trks))):
        this_trk_hits = df.loc[trks[pidx]] #.reset_index()
        if len(this_trk_hits) == 0:
            continue

        this_trk_hits = this_trk_hits.reset_index()
        nwires = this_trk_hits.I2.points.wire.max() - this_trk_hits.I2.points.wire.min()
        nidxs = this_trk_hits.index.max() - this_trk_hits.index.min()
        if (nwires != nidxs):
            this_trk_hits["wireskip"] = np.abs(nwires - nidxs)

        trk_hitdfs.append(this_trk_hits)

    df = pd.concat(trk_hitdfs)
    return df

In [ ]:
plane = 2
proton_hits["plane"] = plane

print(len(proton_hits))
proton_hits = proton_hits[proton_hits.flipped == False]
proton_hits = drop_endhits(proton_hits, ndrop=3, plane=plane)
proton_hits = proton_hits[proton_hits.I2.points.rr > 0.6]

# calculate additional cleanliness info
proton_hits = check_badorder(proton_hits)
proton_hits = check_ntpcs(proton_hits)
proton_hits = check_wireskip(proton_hits)

In [ ]:
proton_hits = proton_hits[(proton_hits.I2.points.pitch < 1) & (proton_hits.badorder == False)] # & (proton_hits.ntpcs == 1)] # & (proton_hits.wireskip == 0)]
print(len(proton_hits))

In [ ]:
nbins = 101
rrbins = np.linspace(0,150,nbins)
dqdxbins = np.linspace(0,0.85e4,nbins)

fig, ax = plt.subplots(1,2, figsize=(14,5))
bins = [rrbins, dqdxbins]
ax[0].hist2d(proton_hits.I2.points.rr, proton_hits.I2.points.dqdx, bins=bins, norm=LogNorm())
ax[0].set_xlabel("Residual Range [cm]")
ax[0].set_ylabel("dQ/dx [ADC/cm]")

cut = (proton_hits.flipped == False)
cut = (proton_hits.truth.p.pdg ==2212)
ax[1].hist2d(proton_hits[cut].I2.points.rr, proton_hits[cut].I2.points.dqdx, bins=bins, norm=LogNorm())
ax[1].set_xlabel("Residual Range [cm]")
ax[1].set_ylabel("dQ/dx [ADC/cm]")

if save:
    plt.savefig("dqdx_rr_postclean.pdf", bbox_inches="tight")
plt.show();

# Flatten

In [ ]:
proton_hits_selected = proton_hits.loc[:, [
    ("selection", "", "", ""),
    ("run", "", "", ""),
    ("subrun", "", "", ""),
    ("evt", "", "", ""),
    ("I2", "points", "tpc", ""),
    ("plane", "", "", ""),
    ("I2", "points", "rr", ""),
    ("I2", "points", "dqdx", ""),
    ("I2", "points", "integral", ""),
    ("I2", "points", "sumadc", ""),
    ("I2", "points", "x", ""),
    ("I2", "points", "y", ""),
    ("I2", "points", "z", ""),
    ("I2", "points", "t", ""),
    ("I2", "points", "pitch", ""),
    ("I2", "points", "phi", ""),
    ("I2", "points", "efield", ""),
    ("trk", "chi2pid", "I2", "chi2_muon"),
    ("trk", "chi2pid", "I2", "chi2_proton")
]].copy()

# Flatten 
proton_hits_selected.columns = [
    "selection",
    "run",
    "subrun",
    "evt",
    "tpc",
    "plane",
    "rr",
    "dqdx",
    "integral",
    "sumadc",
    "x",
    "y",
    "z",
    "t",
    "pitch",
    "phi",
    "efield",
    "trk_chi2_mu",
    "trk_chi2_p"
]
proton_hits_selected = proton_hits_selected.reset_index()
print(len(proton_hits_selected))
proton_hits_selected.head()

In [ ]:
recreate = False

output = "outputs/proton_hits_{}.h5".format(save_tag)
if recreate:
    # open the file, load df, concat, then save again
    with pd.HDFStore(output) as hdf_pd:
        this_key = f"hits"
        df = hdf_pd.get(this_key)
        print("existing df length", len(df))
        df = pd.concat([df, proton_hits_selected], ignore_index=True)
        print("new df length", len(df))
        hdf_pd.put(key=this_key, value=df, format="fixed")
        print(f"Saved {this_key}: {df.memory_usage(deep=True).sum() / (1024**3):.4f} GB")

else:
    with pd.HDFStore(output) as hdf_pd:
        this_key = f"hits"
        hdf_pd.put(key=this_key, value=proton_hits_selected, format="fixed")
        print(f"Saved {this_key}: {proton_hits_selected.memory_usage(deep=True).sum() / (1024**3):.4f} GB")